# Intent Classification Service (Colab + Pinggy)

Notebook này chạy model `ngbaoan/intent-banking` trên GPU của Colab
và expose ra ngoài qua **Pinggy SSH tunnel** — tương tự cách Ollama được serve.

**Luồng:**
1. Load model `ngbaoan/intent-banking` (Unsloth, GPU T4)
2. Khởi động FastAPI server trên port `8001`
3. Mở Pinggy tunnel → copy URL dán vào `INTENT_SERVICE_URL` ở máy local

** Lưu ý Pinggy:** Khi chạy lệnh SSH, **NHẤN PHẢI CHUỘT rồi chọn COPY** để copy URL.
Nếu bấm `Ctrl+C` sẽ tắt kết nối tunnel!

In [ ]:
# ===========================================================
# Cell 1: Cài đặt dependencies
# ===========================================================
!pip install -q unsloth
!pip install -q "fastapi[all]" uvicorn nest-asyncio
!pip install -q transformers peft accelerate

In [ ]:
# ===========================================================
# Cell 2: Load model ngbaoan/intent-banking
# ===========================================================
import torch
import re

MODEL_NAME = "ngbaoan/intent-banking"
MAX_SEQ_LENGTH = 512

PROMPT_TEMPLATE = """Instruct: Classify the following banking query into the correct intent.
Query: {query}
Intent: """

model = None
tokenizer = None

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print(" Model loaded via Unsloth (GPU mode)")
except Exception as e:
    print(f"  Unsloth failed ({e}), falling back to transformers+peft...")
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-7B", torch_dtype=dtype,
        device_map="auto" if device == "cuda" else None,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, MODEL_NAME)
    model.eval()
    print(f" Model loaded via transformers+peft ({device} mode)")

print("\nModel is ready!")

In [ ]:
# ===========================================================
# Cell 3: Hàm inference + test nhanh
# ===========================================================

def predict_intent(message: str) -> dict:
    prompt = PROMPT_TEMPLATE.format(query=message)
    device = next(model.parameters()).device
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            use_cache=True,
            temperature=0.1,
            do_sample=False,
        )

    full_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    intent = "unknown"
    if "Intent:" in full_output:
        raw = full_output.split("Intent:")[-1].strip()
        parsed = raw.split("\n")[0].strip()
        parsed = re.sub(r'[^\w_].*$', '', parsed).strip()
        if parsed:
            intent = parsed

    output_len = outputs.shape[1] - inputs["input_ids"].shape[1]
    confidence = 0.92 if output_len <= 5 else 0.80 if output_len <= 10 else 0.65 if output_len <= 20 else 0.50

    return {"intent": intent, "confidence": confidence}


# Test nhanh
test = predict_intent("My card was stolen and I need to block it immediately.")
print(f"Test result: {test}")

In [ ]:
# ===========================================================
# Cell 4: Khởi động FastAPI server (port 8001)
# ===========================================================
import nest_asyncio
import uvicorn
import threading
import time
from typing import Annotated
from fastapi import FastAPI, Body
from pydantic import BaseModel

nest_asyncio.apply()

intent_app = FastAPI(title="Intent Classification Service")

class IntentRequest(BaseModel):
    message: str

class IntentResponse(BaseModel):
    intent: str
    confidence: float
    reason: str

@intent_app.get("/health")
def health() -> dict:
    return {"status": "ok", "model": MODEL_NAME}

@intent_app.post("/classify")
def classify(request: Annotated[IntentRequest, Body()]) -> IntentResponse:
    result = predict_intent(request.message)
    return IntentResponse(
        intent=result["intent"],
        confidence=result["confidence"],
        reason=f"Classified by {MODEL_NAME} (fine-tuned Qwen2.5-7B on BANKING77)",
    )

PORT = 8001

def run_server():
    uvicorn.run(intent_app, host="0.0.0.0", port=PORT)

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)
print(f" Intent API server running on port {PORT}")
print(f"   Local test: http://localhost:{PORT}/health")

In [ ]:
!pip install -q pinggy
import pinggy
import time
tunnel = pinggy.start_tunnel(forwardto="localhost:8001")
print(f"\n>>> YOUR PINGGY URL IS: {tunnel.urls} <<<\n")
# Giữ cho tunnel hoạt động
while True:
    time.sleep(60)